In [1]:
import torch
import os
os.chdir('../../')

In [2]:
from scipy import linalg
import numpy as np
import os, torch
from tqdm import tqdm
from reports.util import load_config


@torch.no_grad()
def _trace_sqrtm_product(C1: torch.Tensor, C2: torch.Tensor) -> torch.Tensor:
    # Tr sqrtm(C1 @ C2) = Tr sqrt( C1^{1/2} C2 C1^{1/2} )
    s, U = torch.linalg.eigh(C1)                 # C1 = U diag(s) U^T
    s = s.clamp_min(0)
    C1h = (U * s.sqrt()) @ U.t()                 # C1^{1/2}
    M   = C1h @ C2 @ C1h
    w   = torch.linalg.eigvalsh((M + M.t()) * 0.5).clamp_min(0)
    return w.sqrt().sum()

@torch.no_grad()
def calc_fid_stats(mu1, sigma1, mu2, sigma2, eps: float = 1e-6) -> float:
    # 모두 float64 + 동일 device로 정렬
    C1 = torch.as_tensor(sigma1, dtype=torch.float64)
    device = C1.device
    C2 = torch.as_tensor(sigma2, dtype=torch.float64).to(device)
    m1 = torch.as_tensor(mu1,    dtype=torch.float64).to(device).flatten()
    m2 = torch.as_tensor(mu2,    dtype=torch.float64).to(device).flatten()

    D = m1.numel()
    I = torch.eye(D, dtype=torch.float64, device=device)

    # 대칭화 + 정칙화
    C1 = (C1 + C1.t()) * 0.5 + eps * I
    C2 = (C2 + C2.t()) * 0.5 + eps * I

    diff = m1 - m2
    tr_covmean = _trace_sqrtm_product(C1, C2)
    fid = diff.dot(diff) + torch.trace(C1) + torch.trace(C2) - 2.0 * tr_covmean
    return float(fid)

@torch.no_grad()
def calc_fid_pt_dir(pt_dir: str, mu, sigma, eps: float = 1e-6, num=100000, key="inception_feature") -> float:
    # pt_dir에서 'inception_feature'를 모아서 mu1, sigma1 추정 후 FID 계산
    X = []
    for f in tqdm(os.listdir(pt_dir)[:num]):
        if f.endswith(".pt"):
            v = torch.load(os.path.join(pt_dir, f), map_location="cpu").get(key)
            if v is not None:
                X.append(torch.as_tensor(v, dtype=torch.float64).flatten())
    if len(X) < 2:
        raise ValueError("need >=2 features")

    X   = torch.stack(X, 0)                 # [N, D]
    mu1 = X.mean(0)
    Xc  = X - mu1
    sigma1 = (Xc.t() @ Xc) / (X.shape[0] - 1)  # 불편추정

    return calc_fid_stats(mu1, sigma1, mu, sigma, eps=eps)
    #return calculate_frechet_distance(mu1, sigma1, mu, sigma, eps=eps)

def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """Numpy implementation of the Frechet Distance.
    The Frechet distance between two multivariate Gaussians X_1 ~ N(mu_1, C_1)
    and X_2 ~ N(mu_2, C_2) is
            d^2 = ||mu_1 - mu_2||^2 + Tr(C_1 + C_2 - 2*sqrt(C_1*C_2)).

    Stable version by Dougal J. Sutherland.

    Params:
    -- mu1   : Numpy array containing the activations of a layer of the
               inception net (like returned by the function 'get_predictions')
               for generated samples.
    -- mu2   : The sample mean over activations, precalculated on an
               representative data set.
    -- sigma1: The covariance matrix over activations for generated samples.
    -- sigma2: The covariance matrix over activations, precalculated on an
               representative data set.

    Returns:
    --   : The Frechet Distance.
    """

    mu1 = np.atleast_1d(mu1)
    mu2 = np.atleast_1d(mu2)

    sigma1 = np.atleast_2d(sigma1)
    sigma2 = np.atleast_2d(sigma2)

    assert mu1.shape == mu2.shape, \
        'Training and test mean vectors have different lengths'
    assert sigma1.shape == sigma2.shape, \
        'Training and test covariances have different dimensions'

    diff = mu1 - mu2

    # Product might be almost singular
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        msg = ('fid calculation produces singular product; '
               'adding %s to diagonal of cov estimates') % eps
        print(msg)
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    # Numerical error might give slight imaginary component
    if np.iscomplexobj(covmean):
        if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
            m = np.max(np.abs(covmean.imag))
            raise ValueError('Imaginary component {}'.format(m))
        covmean = covmean.real

    tr_covmean = np.trace(covmean)

    return (diff.dot(diff) + np.trace(sigma1)
            + np.trace(sigma2) - 2 * tr_covmean)

def get_clip_score(pt_file, key):
    data = torch.load(pt_file)
    return float(data[key])

from pathlib import Path
import numpy as np
from tqdm import tqdm

def get_clip_scores(dir):
    scores = {}
    
    for pt_file in tqdm(Path(dir).rglob('*.pt')):
        data = torch.load(pt_file)
        for key in data.keys():
            if key.startswith('clip_score'):
                clip_score = get_clip_score(pt_file, key)
                if key in scores:
                    scores[key].append(clip_score)
                else:
                    scores[key] = [clip_score]
    for key in scores.keys():
        scores[key] = np.mean(scores[key])
    return scores
        

In [3]:
pt_dirs = [
        # 'samplings/PixArt-Alpha/3.5/9/Euler/30000/euler_mjhq_0',
        # 'samplings/PixArt-Alpha/3.5/8/Euler/30000/euler_mjhq_0',
        # 'samplings/PixArt-Alpha/3.5/7/Euler/30000/euler_mjhq_0',
        # 'samplings/PixArt-Alpha/3.5/6/Euler/30000/euler_mjhq_0',
        'samplings/PixArt-Alpha/3.5/5/Euler/30000/euler_mjhq_0',
        'samplings/PixArt-Alpha/3.5/4/Euler/30000/euler_mjhq_0',
        'samplings/PixArt-Alpha/3.5/3/Euler/30000/euler_mjhq_0',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mjhq_fid/mjhq_30k_fid_stats.pt')
    print(pt_dir)

    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print('FID :', fid)

    scores = get_clip_scores(pt_dir)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

samplings/PixArt-Alpha/3.5/5/Euler/30000/euler_mjhq_0


100%|██████████| 30001/30001 [00:27<00:00, 1102.97it/s]


FID : 25.854085196795268


30000it [01:47, 278.91it/s]


clip_score_ViT-L/14 0.2689
clip_score_ViT-L/14@336px 0.2746
clip_score_RN101 0.4883
samplings/PixArt-Alpha/3.5/4/Euler/30000/euler_mjhq_0


...

clip_score_ViT-L/14 0.2516
clip_score_ViT-L/14@336px 0.2561
clip_score_RN101 0.4735
samplings/PixArt-Alpha/3.5/3/Euler/30000/euler_mjhq_0


100%|██████████| 30001/30001 [00:31<00:00, 965.24it/s] 


FID : 71.75672231841952


30000it [01:33, 319.71it/s]


clip_score_ViT-L/14 0.2174
clip_score_ViT-L/14@336px 0.2202
clip_score_RN101 0.4456


In [5]:
pt_dirs = [
        'samplings/PixArt-Alpha/3.5/9/DPM-Solver/30000/dpm_mjhq_0/',
        'samplings/PixArt-Alpha/3.5/8/DPM-Solver/30000/dpm_mjhq_0/',
        'samplings/PixArt-Alpha/3.5/7/DPM-Solver/30000/dpm_mjhq_0/',
        'samplings/PixArt-Alpha/3.5/6/DPM-Solver/30000/dpm_mjhq_0/',
        'samplings/PixArt-Alpha/3.5/5/DPM-Solver/30000/dpm_mjhq_0/',
        'samplings/PixArt-Alpha/3.5/4/DPM-Solver/30000/dpm_mjhq_0/',
        'samplings/PixArt-Alpha/3.5/3/DPM-Solver/30000/dpm_mjhq_0/',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mjhq_fid/mjhq_30k_fid_stats.pt')
    print(pt_dir)

    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print('FID :', fid)

    scores = get_clip_scores(pt_dir)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

samplings/PixArt-Alpha/3.5/9/DPM-Solver/30000/dpm_mjhq_0/


  0%|          | 0/30001 [00:00<?, ?it/s]

100%|██████████| 30001/30001 [00:28<00:00, 1039.87it/s]


FID : 7.67496929177878


30000it [01:31, 326.58it/s]


...

clip_score_ViT-L/14 0.2591
clip_score_ViT-L/14@336px 0.2645
clip_score_RN101 0.4793
samplings/PixArt-Alpha/3.5/3/DPM-Solver/30000/dpm_mjhq_0/


100%|██████████| 30001/30001 [00:30<00:00, 974.94it/s] 


FID : 67.82473307880696


30000it [01:30, 329.70it/s]

clip_score_ViT-L/14 0.2145
clip_score_ViT-L/14@336px 0.2181
clip_score_RN101 0.4434


In [3]:
pt_dirs = [
        # 'samplings/PixArt-Alpha/3.5/9/DS-Solver_DDPM/30000/ds_mjhq_0/',
        # 'samplings/PixArt-Alpha/3.5/8/DS-Solver_DDPM/30000/ds_mjhq_0/',
        # 'samplings/PixArt-Alpha/3.5/7/DS-Solver_DDPM/30000/ds_mjhq_0/',
        # 'samplings/PixArt-Alpha/3.5/6/DS-Solver_DDPM/30000/ds_mjhq_0/',
        'samplings/PixArt-Alpha/3.5/5/DS-Solver_DDPM/30000/ds_mjhq_0/',
        'samplings/PixArt-Alpha/3.5/4/DS-Solver_DDPM/30000/ds_mjhq_0/',
        'samplings/PixArt-Alpha/3.5/3/DS-Solver_DDPM/30000/ds_mjhq_0/',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mjhq_fid/mjhq_30k_fid_stats.pt')
    print(pt_dir)

    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print('FID :', fid)

    scores = get_clip_scores(pt_dir)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

samplings/PixArt-Alpha/3.5/5/DS-Solver_DDPM/30000/ds_mjhq_0/


100%|██████████| 30001/30001 [00:24<00:00, 1220.35it/s]


FID : 30.64883627837787


30000it [01:27, 342.19it/s]


clip_score_ViT-L/14 0.2668
clip_score_ViT-L/14@336px 0.2710
clip_score_RN101 0.4856
samplings/PixArt-Alpha/3.5/4/DS-Solver_DDPM/30000/ds_mjhq_0/


...

clip_score_ViT-L/14 0.2493
clip_score_ViT-L/14@336px 0.2505
clip_score_RN101 0.4685
samplings/PixArt-Alpha/3.5/3/DS-Solver_DDPM/30000/ds_mjhq_0/


100%|██████████| 30001/30001 [00:28<00:00, 1051.78it/s]


FID : 97.56857076926292


30000it [01:43, 290.23it/s]

clip_score_ViT-L/14 0.2150
clip_score_ViT-L/14@336px 0.2147
clip_score_RN101 0.4393


In [3]:
pt_dirs = [
        #'samplings/PixArt-Alpha/3.5/9/BNS-Solver_Sep/30000/bns_mjhq_0/',
        'samplings/PixArt-Alpha/3.5/8/BNS-Solver_Sep/30000/bns_mjhq_0/',
        'samplings/PixArt-Alpha/3.5/7/BNS-Solver_Sep/30000/bns_mjhq_0/',
        'samplings/PixArt-Alpha/3.5/6/BNS-Solver_Sep/30000/bns_mjhq_0/',
        'samplings/PixArt-Alpha/3.5/5/BNS-Solver_Sep/30000/bns_mjhq_0/',
        'samplings/PixArt-Alpha/3.5/4/BNS-Solver_Sep/30000/bns_mjhq_0/',
        'samplings/PixArt-Alpha/3.5/3/BNS-Solver_Sep/30000/bns_mjhq_0/',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mjhq_fid/mjhq_30k_fid_stats.pt')
    print(pt_dir)

    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print('FID :', fid)

    scores = get_clip_scores(pt_dir)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

samplings/PixArt-Alpha/3.5/8/BNS-Solver_Sep/30000/bns_mjhq_0/


100%|██████████| 30001/30001 [00:29<00:00, 1033.89it/s]


FID : 12.77846777022296


30000it [01:39, 301.35it/s]


clip_score_ViT-L/14 0.2764
clip_score_ViT-L/14@336px 0.2848
clip_score_RN101 0.4992
samplings/PixArt-Alpha/3.5/7/BNS-Solver_Sep/30000/bns_mjhq_0/


...

FID : 22.168200680917437


29745it [01:34, 313.53it/s]


clip_score_ViT-L/14 0.2716
clip_score_ViT-L/14@336px 0.2797
clip_score_RN101 0.4938
samplings/PixArt-Alpha/3.5/5/BNS-Solver_Sep/30000/bns_mjhq_0/


100%|██████████| 1/1 [00:00<00:00, 11397.57it/s]


ValueError: need >=2 features